# l6e: Budget Enforcement for AI Agent Pipelines

`l6e` wraps your existing LLM calls and enforces a budget across the entire run.

- **Enforcement** — raise `BudgetExceeded` before a call goes out, or reroute to a cheaper model
- **Visibility** — `ctx.budget_status()` gives your agent its own spend state mid-run
- **Zero pipeline changes** — attach via `ctx.call()` or a LangChain callback handler

In [1]:
# !pip install 'l6e[langchain]' langchain-core litellm
# After installing or updating l6e, restart the kernel before running the cells below.

## 1 — Hard enforcement: stop before the call goes out

Set a budget. l6e checks it before every call. When the budget is exhausted, `BudgetExceeded` is raised — before the network request is made.

In [2]:
import l6e

# Stand-in for any real LLM client (litellm, openai, anthropic, ...)
def gpt4o(model, messages):
    class R:
        class usage:
            prompt_tokens = 850
            completion_tokens = 120
    return R()

policy = l6e.PipelinePolicy(budget=0.001, budget_mode=l6e.BudgetMode.HALT)

with l6e.pipeline("demo", policy) as ctx:
    # First call: retrieval — costs $0.00333, already 3× over the $0.001 budget
    ctx.call(fn=gpt4o, model="gpt-4o",
             messages=[{"role": "user", "content": "What do AI agents cost?"}],
             stage="retrieval")

    print(f"After retrieval: spent ${ctx.budget_status().spent_usd:.5f} of ${policy.budget:.3f}")

    # Second call: l6e blocks it before the network request is made
    try:
        ctx.call(fn=gpt4o, model="gpt-4o",
                 messages=[{"role": "user", "content": "Analyze cost drivers."}],
                 stage="reasoning")
    except l6e.BudgetExceeded as e:
        print(f"\nBlocked: {e}")
        print(f"  e.spent:  ${e.spent:.5f}")
        print(f"  e.budget: ${e.budget:.5f}")

After retrieval: spent $0.00333 of $0.001

Blocked: Budget exceeded: spent $0.003325 of $0.001000 budget. budget_pressure:halt
  e.spent:  $0.00333
  e.budget: $0.00100


## 2 — Agent self-regulation: `ctx.budget_status()`

No gateway, no observability tool, and no framework gives the agent itself visibility into its own spend mid-run. `ctx.budget_status()` is the only primitive that does this.

The agent calls it at any point — zero tokens, pure arithmetic — and branches on `budget_pressure` (`low` / `moderate` / `high` / `critical`). This is different from HALT: the agent decides what to do, not the enforcement layer.

In [3]:
def reasoning(model, messages):
    class R:
        class usage:
            prompt_tokens = 620
            completion_tokens = 380
    return R()

policy = l6e.PipelinePolicy(budget=0.004, budget_mode=l6e.BudgetMode.REROUTE)

with l6e.pipeline("adapt", policy) as ctx:
    # Retrieval runs — costs $0.00333, consuming 83% of the $0.004 budget
    ctx.call(fn=gpt4o, model="gpt-4o",
             messages=[{"role": "user", "content": "What do AI agents cost?"}],
             stage="retrieval")

    status = ctx.budget_status()
    print(f"After retrieval: {status.pct_used:.0f}% used — pressure: {status.budget_pressure}")
    print(f"  spent ${status.spent_usd:.5f} / remaining ${status.remaining_usd:.5f}")

    if status.budget_pressure in ("high", "critical"):
        print("\nAgent self-regulated: skipped deep reasoning, returned partial result")
    else:
        ctx.call(fn=reasoning, model="gpt-4o",
                 messages=[{"role": "user", "content": "Analyze cost drivers."}],
                 stage="reasoning")
        print("Full reasoning completed.")

After retrieval: 83% used — pressure: high
  spent $0.00333 / remaining $0.00067

Agent self-regulated: skipped deep reasoning, returned partial result


The agent read its own budget state and chose to skip the expensive next step — no exception, no external enforcement.

## 3 — LangChain: `L6eCallbackHandler`

For existing LangChain pipelines, attach a callback handler — no other changes required.

```python
handler = L6eCallbackHandler(ctx)
chain.invoke(input, config={"callbacks": [handler]})
```

l6e infers the pipeline stage automatically from prompt complexity — no tags, no `with_config` boilerplate at every call site. The cell below uses `FakeLocalRouter` to simulate Ollama without hardware; everything else is real LangChain.

In [4]:
import warnings
warnings.filterwarnings("ignore")

from langchain_core.language_models.fake import FakeListLLM
from langchain_core.prompts import PromptTemplate

from l6e.adapters.langchain import L6eCallbackHandler
from l6e.costs import LiteLLMCostEstimator

MODEL_FRONTIER = "gpt-4o"
MODEL_LOCAL    = "ollama/qwen2.5:7b"
STAGE_TOKENS   = {"retrieval": (850, 120), "reasoning": (620, 380), "formatting": (430, 95)}

estimator = LiteLLMCostEstimator()
baseline  = sum(estimator.estimate(MODEL_FRONTIER, p, c) for p, c in STAGE_TOKENS.values())

CHAINS = [
    ("retrieval",  "List the top AI agent frameworks."),
    ("formatting", "What are the main cost drivers in LLM-powered agent pipelines?"),
    ("reasoning",  "Step 1: analyze cost drivers. Step 2: evaluate trade-offs. Step 3: recommend."),
]


class FakeLocalRouter:
    """Simulates Ollama without hardware — only used here so the demo runs without a GPU."""
    def best_local_model(self): return MODEL_LOCAL


class TokenAwareFakeLLM(FakeListLLM):
    stage: str = "retrieval"
    model_name: str = MODEL_FRONTIER

    @property
    def _llm_type(self): return self.model_name

    @property
    def _identifying_params(self): return {"model": self.model_name, **super()._identifying_params}

    def _generate(self, prompts, stop=None, run_manager=None, **kw):
        result = super()._generate(prompts, stop=stop, run_manager=run_manager, **kw)
        p, c = STAGE_TOKENS.get(self.stage, (500, 150))
        result.llm_output = {"token_usage": {"prompt_tokens": p, "completion_tokens": c}}
        return result


lc_policy = l6e.PipelinePolicy(
    budget=0.02, budget_mode=l6e.BudgetMode.REROUTE,
    stage_routing={
        "retrieval":  l6e.StageRoutingHint.LOCAL,
        "reasoning":  l6e.StageRoutingHint.CLOUD_FRONTIER,
        "formatting": l6e.StageRoutingHint.CLOUD_STANDARD,
    },
)

with l6e.pipeline("lc-demo", lc_policy, router=FakeLocalRouter()) as lc_ctx:
    handler = L6eCallbackHandler(lc_ctx)
    cfg     = {"callbacks": [handler]}

    for stage, prompt in CHAINS:
        chain = PromptTemplate.from_template("{q}") | TokenAwareFakeLLM(responses=["ok"], stage=stage)
        chain.invoke({"q": prompt}, config=cfg)

s = lc_ctx.run_summary()
print(f"{'Stage':<14}  {'Inferred':<10}  {'Requested':<10}  {'Used':<26}  {'Cost':>9}  Rerouted")
print("-" * 82)
for r in s.records:
    print(f"{r.stage or '?':<14}  {'auto':<10}  {r.model_requested:<10}  {r.model_used:<26}  ${r.cost_usd:.5f}  {'YES ←' if r.rerouted else '-'}")
print("-" * 82)
print(f"Baseline (all gpt-4o): ${baseline:.5f}")
print(f"With l6e:              ${s.total_cost:.5f}")
print(f"Savings:               ${s.savings_usd:.5f}  ({s.savings_usd / baseline * 100:.1f}% reduction)")

Stage           Inferred    Requested   Used                             Cost  Rerouted
----------------------------------------------------------------------------------
retrieval       auto        gpt-4o      ollama/qwen2.5:7b           $0.00000  YES ←
formatting      auto        gpt-4o      gpt-4o                      $0.00202  -
reasoning       auto        gpt-4o      gpt-4o                      $0.00535  -
----------------------------------------------------------------------------------
Baseline (all gpt-4o): $0.01070
With l6e:              $0.00738
Savings:               $0.00333  (31.1% reduction)


If you want deterministic routing for a specific chain — overriding inference — declare the stage explicitly with a tag:

```python
chain.with_config(tags=["l6e_stage:retrieval"]).invoke(input, config=cfg)
```

In [5]:
with l6e.pipeline("lc-explicit", lc_policy, router=FakeLocalRouter()) as lc_ctx2:
    handler2 = L6eCallbackHandler(lc_ctx2)
    cfg2     = {"callbacks": [handler2]}

    for stage, prompt in CHAINS:
        chain = PromptTemplate.from_template("{q}") | TokenAwareFakeLLM(responses=["ok"], stage=stage)
        chain.with_config(tags=[f"l6e_stage:{stage}"]).invoke({"q": prompt}, config=cfg2)

s2 = lc_ctx2.run_summary()
print(f"{'Stage':<14}  {'Source':<10}  {'Requested':<10}  {'Used':<26}  {'Cost':>9}  Rerouted")
print("-" * 82)
for r in s2.records:
    print(f"{r.stage or '?':<14}  {'declared':<10}  {r.model_requested:<10}  {r.model_used:<26}  ${r.cost_usd:.5f}  {'YES ←' if r.rerouted else '-'}")

Stage           Source      Requested   Used                             Cost  Rerouted
----------------------------------------------------------------------------------
retrieval       declared    gpt-4o      ollama/qwen2.5:7b           $0.00000  YES ←
formatting      declared    gpt-4o      gpt-4o                      $0.00202  -
reasoning       declared    gpt-4o      gpt-4o                      $0.00535  -


**$0.00333 saved per run. 31.1% reduction. One policy declaration.**

At the volumes where developers start complaining about bills:

| Runs / day | Saved / day | Saved / month |
|------------|-------------|---------------|
| 1,000      | $3.33       | **~$100**      |
| 10,000     | $33         | **~$1,000**    |
| 100,000    | $333        | **~$10,000**   |

A team running 10,000 agent pipeline invocations per day — not unusual for a production RAG system — saves ~$1,000/month with zero changes to pipeline logic. The only addition is a `PipelinePolicy`.

## 4 — Running with a real API key

Set `OPENAI_API_KEY` and run the cell below. `l6e.pipeline()` handles all wiring.
Swap `CLOUD_STANDARD` to `LOCAL` on the retrieval line to route through a running Ollama instance.

In [6]:
import os, litellm

if not os.environ.get("OPENAI_API_KEY"):
    print("Set OPENAI_API_KEY to run this cell.")
else:
    real_policy = l6e.PipelinePolicy(
        budget=0.05, budget_mode=l6e.BudgetMode.REROUTE,
        stage_routing={"retrieval": l6e.StageRoutingHint.CLOUD_STANDARD},  # swap to LOCAL for Ollama
    )
    with l6e.pipeline("real", real_policy) as ctx:
        r = ctx.call(
            fn=lambda model, msgs: litellm.completion(model=model, messages=msgs),
            model="gpt-4o",
            messages=[{"role": "user", "content": "What drives AI agent pipeline costs?"}],
            stage="retrieval",
        )
        status = ctx.budget_status()
        print(f"Model:    {r.model}")
        print(f"Tokens:   {r.usage.prompt_tokens}p + {r.usage.completion_tokens}c")
        print(f"Cost:     ${status.spent_usd:.5f}")
        print(f"Pressure: {status.budget_pressure}")

Set OPENAI_API_KEY to run this cell.


## Run log

Every run is appended to `.l6e/runs.jsonl` on context exit — automatically, no extra code.

In [7]:
import json
from pathlib import Path

log = Path(".l6e/runs.jsonl")
if log.exists():
    lines = log.read_text().strip().splitlines()
    print(f".l6e/runs.jsonl — {len(lines)} run(s) recorded\n")
    print(json.dumps(json.loads(lines[-1]), indent=2))
else:
    print("No run log found — run the cells above first.")

.l6e/runs.jsonl — 98 run(s) recorded

{
  "run_id": "lc-explicit",
  "policy": {
    "budget": 0.02,
    "budget_mode": "reroute",
    "on_budget_exceeded": "raise",
    "fallback_result": null,
    "latency_sla": null,
    "reroute_threshold": 0.8,
    "stage_routing": {
      "retrieval": "local",
      "reasoning": "cloud_frontier",
      "formatting": "cloud_standard"
    },
    "stage_overrides": {}
  },
  "total_cost": 0.0073750000000000005,
  "calls_made": 3,
  "reroutes": 1,
  "savings_usd": 0.0033250000000000007,
  "records": [
    {
      "call_index": 0,
      "model_requested": "gpt-4o",
      "model_used": "ollama/qwen2.5:7b",
      "prompt_tokens": 850,
      "completion_tokens": 120,
      "cost_usd": 0,
      "rerouted": true,
      "elapsed_ms": 0.24470902280882,
      "stage": "retrieval",
      "prompt_complexity": null,
      "is_multi_turn": false
    },
    {
      "call_index": 1,
      "model_requested": "gpt-4o",
      "model_used": "gpt-4o",
      "prompt_toke